<a href="https://colab.research.google.com/github/1hass1/My-Own-GPT/blob/main/My_own_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [ ]:
# Imports
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Download Tiny Shakespeare
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-01-09 06:06:59--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-01-09 06:07:00 (28.9 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [ ]:
# This allows easy Hyperparameter experimentation
class GPTConfig:
  def __init__(self, vocab_size, block_size,
               n_layers=4, n_heads=4, embedding_size=256, dropout=0.1):
    self.vocab_size = vocab_size
    self.block_size = block_size
    self.n_layers = n_layers
    self.n_heads = n_heads
    self.embedding_size = embedding_size
    self.dropout = dropout

In [ ]:
# Multi-head self attention

class CausalSelfAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    assert config.embedding_size % config.n_heads == 0, (
    f"embedding_size ({config.embedding_size}) must be divisible by n_heads ({config.n_heads})")
    self.n_heads = config.n_heads
    self.head_dim = config.embedding_size // config.n_heads
    self.key = nn.Linear(config.embedding_size, config.embedding_size)
    self.query = nn.Linear(config.embedding_size, config.embedding_size)
    self.value = nn.Linear(config.embedding_size, config.embedding_size)
    self.proj = nn.Linear(config.embedding_size, config.embedding_size)
    self.attn_dropout = nn.Dropout(config.dropout)
    self.resid_dropout = nn.Dropout(config.dropout)
    # Causal mask
    self.register_buffer("mask", torch.tril(torch.ones(config.block_size, config.block_size)))

  def forward(self,x):
    B, T, C = x.shape
    k = self.key(x).view(B, T, self.n_heads, self.head_dim).transpose(1,2)
    q = self.query(x).view(B, T, self.n_heads, self.head_dim).transpose(1,2)
    v = self.value(x).view(B, T, self.n_heads, self.head_dim).transpose(1,2)
    att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
    att = att.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
    att = F.softmax(att, dim=-1)
    att = self.attn_dropout(att)
    y = att @ v
    y = y.transpose(1,2).contiguous().view(B, T, C)
    y = self.resid_dropout(self.proj(y))
    return y




In [ ]:
# Transformer block

class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ln1 = nn.LayerNorm(config.embedding_size)
    self.ln2 = nn.LayerNorm(config.embedding_size)
    self.attn = CausalSelfAttention(config)
    self.mlp = nn.Sequential(
        nn.Linear(config.embedding_size, 4 * config.embedding_size),
        nn.GELU(),
        nn.Linear(4 * config.embedding_size, config.embedding_size),
        nn.Dropout(config.dropout)
    )

  def forward(self, x):
    x = x + self.attn(self.ln1(x))
    x = x + self.mlp(self.ln2(x))
    return x



In [ ]:
# GPT Model

class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.token_emb = nn.Embedding(config.vocab_size, config.embedding_size)
    self.positional_emb = nn.Embedding(config.block_size, config.embedding_size)
    self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layers)])
    self.ln_f = nn.LayerNorm(config.embedding_size)
    self.head = nn.Linear(config.embedding_size, config.vocab_size, bias=False)
    self.block_size = config.block_size

  def forward(self, idx, targets=None):
    B, T = idx.shape
    pos = torch.arange(0, T, device=idx.device)
    x = self.token_emb(idx) + self.positional_emb(pos)
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.head(x)

    loss = None
    if targets is not None:
      loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

    return logits, loss


In [ ]:
# Dataset (Fine-Tuning on any text file)

class TextDataset(Dataset):
  def __init__(self, text, block_size, tokenizer):
    self.block_size = block_size
    self.data = tokenizer(text)

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, idx):
    x = torch.tensor(self.data[idx:idx+self.block_size])
    y = torch.tensor(self.data[idx+1:idx+self.block_size+1])
    return x, y

In [ ]:
# Simple Tokenizer:

def build_tokenizer(text):
  chars = sorted(list(set(text)))
  stoi = {ch:i for i, ch in enumerate(chars)}
  itos = {i:ch for ch, i in stoi.items()}

  def encode(text):
    return [stoi[c] for c in text]

  def decode(tokens):
    return ''.join([itos[t] for t in tokens])

  return encode, decode, len(chars)

In [ ]:
# Load data

with open("input.txt", "r", encoding="utf-8") as f:
  text = f.read()

print(f"Length of Dataset: {len(text)}")
print(text[:500])

encode, decode, vocab_size = build_tokenizer(text)


Length of Dataset: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
# Initialize model

device = "cuda"

config = GPTConfig(vocab_size=vocab_size, block_size=64,
                   n_layers=4, n_heads=4, embedding_size=256, dropout=0.1)

model = GPT(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
# Fine-tuning

dataset = TextDataset(text, config.block_size, encode)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model.train()
for epoch in range(10):
  pbar = tqdm(loader)
  for x, y in pbar:
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    _, loss = model(x, y)
    loss.backward()
    optimizer.step()
    pbar.set_description(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 1.2778: 100%|██████████| 34855/34855 [11:45<00:00, 49.40it/s]
Epoch 1 | Loss: 1.0466: 100%|██████████| 34855/34855 [11:42<00:00, 49.60it/s]
Epoch 2 | Loss: 0.9613: 100%|██████████| 34855/34855 [11:42<00:00, 49.58it/s]
Epoch 3 | Loss: 0.7679: 100%|██████████| 34855/34855 [11:41<00:00, 49.66it/s]
Epoch 4 | Loss: 0.7413: 100%|██████████| 34855/34855 [11:44<00:00, 49.44it/s]
Epoch 5 | Loss: 1.0021: 100%|██████████| 34855/34855 [11:48<00:00, 49.23it/s]
Epoch 6 | Loss: 0.7603: 100%|██████████| 34855/34855 [11:47<00:00, 49.27it/s]
Epoch 7 | Loss: 0.7971: 100%|██████████| 34855/34855 [11:49<00:00, 49.12it/s]
Epoch 8 | Loss: 1.0042: 100%|██████████| 34855/34855 [11:50<00:00, 49.07it/s]
Epoch 9 | Loss: 0.7747: 100%|██████████| 34855/34855 [11:48<00:00, 49.20it/s]


In [ ]:
# Text Generation

def generate(model, start_text, steps=100):
    model.eval()
    idx = torch.tensor([encode(start_text)], device=device)

    for _ in range(steps):
        idx_cond = idx[:, -config.block_size:]
        logits, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)

    return decode(idx[0].tolist())

print(generate(model, "Math", 200))

Mathan, my lord. Come, away to-night
Is have all that all the world that were,
I then bear your houses: put them on there in another's eye.
Order these days are suitors that will swear
What thy best fealt
